# AareML — 03 LSTM Single Site

**Project:** AareML — Predicting River Water Quality in Swiss Catchments  
**Dataset:** CAMELS-CH-Chem (Nascimento et al., 2025)  
**Reference benchmark:** LakeBeD-US (McAfee et al., 2025)

---

## Architecture

Seq2Seq LSTM matching the LakeBeD-US benchmark design:

```
Encoder LSTM (depth=2)  →  hidden state  →  Decoder LSTM (depth=2)
  input: [lookback, n_feat]                  output: [horizon, n_targets]
```

| Setting | Value |
|---------|-------|
| Lookback | 21 days |
| Forecast horizon | 14 days |
| Targets | DO (mg/L), Temperature (°C) |
| Tuning | Optuna TPE (20 trials) |
| Loss | MSE (scaled space) |
| Optimiser | AdamW + ReduceLROnPlateau |
| Early stopping | patience=12 |
| Metrics | RMSE, MAE, NSE, KGE + 95% bootstrap CIs |


## 0 · Setup


In [ ]:
import sys, random, time
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import StandardScaler

from src.config  import *
from src.data    import load_gauge, load_metadata, preprocess, train_val_test_split, make_windows
from src.metrics import rmse_per_step, metrics_table, block_bootstrap_ci, mean_rmse
from src.model   import (
    RiverDataset, Seq2SeqLSTM,
    train_model, predict, get_y_true, save_checkpoint,
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')


In [ ]:
# ── GPU / DataLoader configuration ────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# On GPU: use 4 workers + pin_memory for faster data transfer to GPU
# On CPU (Mac): use 0 workers (avoids multiprocessing overhead)
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'DataLoader workers: {NUM_WORKERS}, pin_memory: {PIN_MEMORY}')


## 1 · Load and Preprocess


In [ ]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)
train, val, test = train_val_test_split(data)

# Training means used for NaN imputation — computed once, reused for all splits
# Compute training means without duplicated index (FEATURES and TARGETS overlap)
train_means = (
    pd.concat([train[FEATURES].mean(), train[TARGETS].mean()])
    .groupby(level=0).first()   # deduplicate temp_sensor / O2C_sensor
)

print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train):,} days)')
print(f'Val  : {val.index.min().date()}   → {val.index.max().date()}  ({len(val):,} days)')
print(f'Test : {test.index.min().date()}  → {test.index.max().date()}  ({len(test):,} days)')


## 2 · Focus Gauge Metadata


In [ ]:
meta = load_metadata()
# Find the gauge row — try common column name variants
id_col = next((c for c in meta.columns if 'gauge' in c.lower() and 'id' in c.lower()), meta.columns[0])
gauge_row = meta[meta[id_col].astype(str) == str(FOCUS_GAUGE)]
if not gauge_row.empty:
    print(f'Focus gauge: {FOCUS_GAUGE}')
    print(gauge_row.T.to_string())
else:
    print(f'Gauge {FOCUS_GAUGE} — id column used: {id_col}')
    print('Column names:', meta.columns.tolist())


## 3 · Feature Scaling

Scalers are fitted **only on training data** and reused across all splits.
NaN imputation uses training-set means — no information from val/test leaks in.


In [ ]:
# Impute with training means before fitting scaler
def impute(df, means):
    return df.fillna(means)

feat_scaler = StandardScaler().fit(impute(train[FEATURES], train_means[FEATURES]))
tgt_scaler  = StandardScaler().fit(impute(train[TARGETS],  train_means[TARGETS]))

def scale_split(df):
    X = feat_scaler.transform(impute(df[FEATURES], train_means[FEATURES])).astype(np.float32)
    y = tgt_scaler.transform( impute(df[TARGETS],  train_means[TARGETS])).astype(np.float32)
    return X, y

X_tr_s, y_tr_s = scale_split(train)
X_va_s, y_va_s = scale_split(val)
X_te_s, y_te_s = scale_split(test)

print('Feature means  (train):', dict(zip(FEATURES, feat_scaler.mean_.round(3))))
print('Target  scales (train):', dict(zip(TARGETS,  tgt_scaler.scale_.round(3))))


## 4 · Build Windows and Datasets


In [ ]:
X_train, y_train, d_train = make_windows(train, train_means)
X_val,   y_val,   d_val   = make_windows(val,   train_means)
X_test,  y_test,  d_test  = make_windows(test,  train_means)


# Build scaled window arrays separately (scaler applied after windowing)
def scale_windows(X_raw, y_raw):
    N, L, F = X_raw.shape
    X_s = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype(np.float32)
    Nt, H, T = y_raw.shape
    y_s = tgt_scaler.transform(y_raw.reshape(-1, T)).reshape(Nt, H, T).astype(np.float32)
    return X_s, y_s

Xs_train, ys_train = scale_windows(X_train, y_train)
Xs_val,   ys_val   = scale_windows(X_val,   y_val)
Xs_test,  ys_test  = scale_windows(X_test,  y_test)

ds_train = RiverDataset(Xs_train, ys_train)
ds_val   = RiverDataset(Xs_val,   ys_val)
ds_test  = RiverDataset(Xs_test,  ys_test)

print(f'Train windows: {len(ds_train):,}  |  Val: {len(ds_val):,}  |  Test: {len(ds_test):,}')


## 5 · Model Architecture


In [ ]:
# Constants derived from config
N_FEAT = len(FEATURES)  # 4
N_TGT  = len(TARGETS)   # 2

# Sanity check: build a model and verify output shape
_m = Seq2SeqLSTM(N_FEAT, N_TGT, hidden=64, n_layers=2, dropout=0.2)
_x = torch.randn(8, LOOKBACK, N_FEAT)
_y = _m(_x)
total_params = sum(p.numel() for p in _m.parameters())
print(f'Output shape : {_y.shape}  (expected [8, {HORIZON}, {N_TGT}])')
print(f'Parameters   : {total_params:,}')
print(_m)


## 6 · Quick Default Training Run


In [ ]:
BATCH_SIZE = 64
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print('Training default model (hidden=64, layers=2, dropout=0.2, lr=1e-3)...')
t0 = time.time()

default_model = Seq2SeqLSTM(N_FEAT, N_TGT, hidden=64, n_layers=2, dropout=0.2)
default_model, hist_default = train_model(
    default_model, dl_train, dl_val,
    lr=1e-3, epochs=100, patience=12, device=DEVICE, verbose=True,
)
print(f'Training time: {time.time()-t0:.1f}s')


## 7 · Training Curve


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_x = range(1, len(hist_default['train']) + 1)
ax.plot(epochs_x, hist_default['train'], label='Train loss')
ax.plot(epochs_x, hist_default['val'],   label='Val loss',   linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss (scaled targets)')
ax.set_title('LSTM Training Curve — Gauge 2473')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_training_curve.png', bbox_inches='tight')
plt.show()


## 8 · Optuna Hyper-Parameter Tuning

| Parameter | Search space |
|-----------|-------------|
| `hidden` | {32, 64, 128, 256} |
| `n_layers` | 1 – 3 |
| `dropout` | 0.0 – 0.5 |
| `lr` | 1e-4 – 1e-2 (log) |
| `batch_size` | {32, 64, 128} |

Objective: minimum validation MSE (scaled space).


In [ ]:
# ── Hyperparameter search space ────────────────────────────────────────────
N_TRIALS   = 20
BATCH_SIZE = 64   # default fallback

def objective(trial):
    hidden     = trial.suggest_categorical('hidden',     [32, 64, 128, 256])
    n_layers   = trial.suggest_categorical('n_layers',   [1, 2])
    dropout    = trial.suggest_float(      'dropout',     0.0,  0.5)
    lr         = trial.suggest_float(      'lr',          1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])

    dl_tr = DataLoader(ds_train, batch_size=batch_size, shuffle=True,
                       drop_last=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    dl_va = DataLoader(ds_val, batch_size=batch_size, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = Seq2SeqLSTM(N_FEAT, N_TGT, hidden, n_layers, dropout)
    _, history = train_model(
        model, dl_tr, dl_va,
        lr=lr, epochs=60, patience=8, device=DEVICE, verbose=False,
    )
    return min(history['val'])

print(f'Running Optuna ({N_TRIALS} trials)...')
t0 = time.time()
study = optuna.create_study(
    direction='minimize',
    study_name='aareml_gauge2473',
    storage='sqlite:///' + str(Path('../results/optuna_study.db').resolve()),
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Finished in {time.time()-t0:.0f}s')

best = study.best_params
print(f'Best val loss : {study.best_value:.6f}')
print(f'Best params   : {best}')


## 9 · Retrain Best Model

Trained on train+val data. Early stopping uses a **held-out shard of val**
(last 20%) that was never part of the training data — avoids the leakage
that occurs when val is inside the training set.


In [ ]:
# Split val into training extension (80%) and final early-stop monitor (20%)
n_val = len(ds_val)
n_val_stop = max(30, int(n_val * 0.20))
n_val_train = n_val - n_val_stop

from torch.utils.data import Subset
ds_val_train = Subset(ds_val, range(n_val_train))          # used for training
ds_val_stop  = Subset(ds_val, range(n_val_train, n_val))   # used for early stop only

ds_trainval  = ConcatDataset([ds_train, ds_val_train])
dl_trainval  = DataLoader(ds_trainval, batch_size=best['batch_size'],
                          shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val_stop  = DataLoader(ds_val_stop,  batch_size=256, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f'Train+val windows: {len(ds_trainval):,}  |  Early-stop monitor: {len(ds_val_stop):,}')

print('Retraining best model...')
t0 = time.time()
best_model = Seq2SeqLSTM(
    N_FEAT, N_TGT,
    hidden=best['hidden'], n_layers=best['n_layers'], dropout=best['dropout'],
)
best_model, hist_best = train_model(
    best_model, dl_trainval, dl_val_stop,
    lr=best['lr'], epochs=150, patience=15, device=DEVICE, verbose=True,
)
print(f'Done in {time.time()-t0:.1f}s')

RESULTS_DIR.mkdir(exist_ok=True)
save_checkpoint(RESULTS_DIR / 'lstm_single_site_best.pt',
                best_model, best, feat_scaler, tgt_scaler)
print('Checkpoint saved → results/lstm_single_site_best.pt')


## 10 · Test-Set Evaluation


In [ ]:
y_pred_default = predict(default_model, ds_test, tgt_scaler, device=DEVICE)
y_pred_best    = predict(best_model,    ds_test, tgt_scaler, device=DEVICE)

# Ground-truth in original units
y_true_test = get_y_true(ds_test, tgt_scaler)

models_dict = {
    'LSTM (default)': y_pred_default,
    'LSTM (best)':    y_pred_best,
}

# Try to merge with baseline results
lstm_results = metrics_table(models_dict, y_true_test, n_boot=500)
try:
    bl_df = pd.read_csv(RESULTS_DIR / 'baseline_results.csv').rename(columns={'Baseline': 'Model'})
    combined = pd.concat([bl_df, lstm_results], ignore_index=True)
except FileNotFoundError:
    combined = lstm_results
    print('Note: run notebook 02 first to include baseline rows')

print(combined.to_string(index=False))
combined.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
print('\nSaved → results/model_comparison.csv')


## 11 · RMSE by Forecast Horizon


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
steps = np.arange(1, HORIZON + 1)

# Load baselines if available
bl_preds = {}
try:
    bl_df = pd.read_csv(RESULTS_DIR / 'baseline_results.csv')
    print('Baseline results loaded for comparison')
except FileNotFoundError:
    print('Run notebook 02 first for baseline comparison')

model_styles = {
    'LSTM (default)': ('s--', '#BBA8CC'),
    'LSTM (best)':    ('o-',  '#8B4FA8'),
}

for ax_i, (t_i, t) in enumerate(enumerate(TARGETS)):
    ax = axes[ax_i]
    for name, y_pred in models_dict.items():
        marker, color = model_styles[name]
        per = rmse_per_step(y_true_test, y_pred)[:, t_i]
        ax.plot(steps, per, marker, color=color, linewidth=2, label=name, markersize=5)
    ax.set_xlabel('Forecast horizon (days)', fontsize=11)
    ax.set_ylabel('RMSE', fontsize=11)
    ax.set_title(TARGET_LABELS[t], fontsize=12)
    ax.set_xticks(steps)
    ax.legend(fontsize=9)

fig.suptitle('LSTM RMSE by Forecast Horizon — Gauge 2473 (Test Set)',
             fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_rmse_by_horizon.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── I5: Per-horizon RMSE — LSTM vs Ridge baseline ──────────────────────────
import json as _json, pandas as _pd

# Load baseline results for Ridge
baseline_csv = RESULTS_DIR / "baseline_results.csv"
if baseline_csv.exists():
    bl = _pd.read_csv(baseline_csv)
    # baseline_results has columns: Model, Target, RMSE, ... 
    # but we need per-horizon data — load from the horizon CSV if it exists
    horizon_csv = RESULTS_DIR / "baseline_rmse_by_horizon.csv"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_names = ["DO (mg/L)", "Temp (°C)"]

for ax, tname in zip(axes, target_names):
    # LSTM per-horizon RMSE (best model)
    if 'y_pred_best' in dir() and 'y_true_test' in dir():
        tidx = 0 if "DO" in tname else 1
        rmse_per_h = [
            np.sqrt(np.mean((y_pred_best[:, h, tidx] - y_true_test[:, h, tidx])**2))
            for h in range(HORIZON)
        ]
        ax.plot(range(1, HORIZON+1), rmse_per_h, 'o-', color='#01696F',
                linewidth=2, markersize=4, label='LSTM (best Optuna)')

    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel(f"RMSE ({tname})")
    ax.set_title(f"Per-horizon RMSE — {tname}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("I5: LSTM Per-Horizon RMSE — Gauge 2473 Test Set", fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_lstm_rmse_per_horizon_vs_baseline.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_lstm_rmse_per_horizon_vs_baseline.png")


## 12 · Annual RMSE Breakdown

RMSE per calendar year in the test set — shows inter-annual variability
and prevents a single anomalous year from hiding poor generalisation.


In [ ]:
years = sorted(set(d_test.year))
annual_rows = []
for yr in years:
    mask = np.array(d_test.year) == yr
    if mask.sum() < 10:
        continue
    for name, y_pred in models_dict.items():
        for t_i, t in enumerate(TARGETS):
            rmse = np.sqrt(((y_true_test[mask,:,t_i] - y_pred[mask,:,t_i])**2).mean())
            annual_rows.append({'Year': yr, 'Model': name,
                                'Target': TARGET_LABELS[t], 'RMSE': round(rmse, 4)})

annual_df = pd.DataFrame(annual_rows)
print(annual_df.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax_i, tgt_label in enumerate([TARGET_LABELS[t] for t in TARGETS]):
    ax = axes[ax_i]
    sub = annual_df[annual_df['Target'] == tgt_label]
    for name, grp in sub.groupby('Model'):
        _, color = model_styles[name]
        ax.plot(grp['Year'], grp['RMSE'], 'o-', color=color, label=name, linewidth=1.8)
    ax.set_xlabel('Year'); ax.set_ylabel('RMSE'); ax.set_title(tgt_label, fontsize=12)
    ax.legend(fontsize=9)

fig.suptitle('Annual RMSE Breakdown — Gauge 2473', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_annual_rmse.png', bbox_inches='tight')
plt.show()


## 13 · Example Forecast Plots


In [ ]:
def find_nearest_idx(dates, target_date):
    diffs = np.abs((dates - pd.Timestamp(target_date)).days)
    return int(np.argmin(diffs))

winter_idx = find_nearest_idx(d_test, '2018-01-15')
summer_idx = find_nearest_idx(d_test, '2018-07-15')
print(f'Winter window: {d_test[winter_idx].date()}')
print(f'Summer window: {d_test[summer_idx].date()}')

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
steps_arr = np.arange(1, HORIZON + 1)

for row, (label, idx) in enumerate([('Winter 2018', winter_idx), ('Summer 2018', summer_idx)]):
    for col, (t_i, t) in enumerate(enumerate(TARGETS)):
        ax = axes[row][col]
        truth = y_true_test[idx, :, t_i]
        ax.plot(steps_arr, truth, 'k-', linewidth=2, label='Observed')
        for name, y_pred in models_dict.items():
            marker, color = model_styles[name]
            ax.plot(steps_arr, y_pred[idx, :, t_i], marker,
                    color=color, label=name, linewidth=1.5)
        ax.set_title(f'{label} — {TARGET_LABELS[t]}', fontsize=10)
        ax.set_xlabel('Forecast day', fontsize=9)
        ax.set_ylabel(TARGET_LABELS[t], fontsize=9)
        ax.legend(fontsize=8)

fig.suptitle('LSTM Forecast Examples — Gauge 2473', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_example_forecasts.png', bbox_inches='tight')
plt.show()


## 14 · Residual Diagnostics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax_i, (t_i, t) in enumerate(enumerate(TARGETS)):
    ax = axes[ax_i]
    obs  = y_true_test[:, :, t_i].ravel()
    pred = y_pred_best[:, :, t_i].ravel()
    res  = obs - pred
    ax.scatter(pred, res, alpha=0.12, s=4, color='#8B4FA8', rasterized=True)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel(f'Predicted {TARGET_LABELS[t]}', fontsize=10)
    ax.set_ylabel('Residual', fontsize=10)
    bias = res.mean()
    ax.set_title(f'{TARGET_LABELS[t]} — bias={bias:+.3f}', fontsize=11)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_residuals.png', bbox_inches='tight')
plt.show()


## 15 · Optuna Visualisation


In [ ]:
try:
    from optuna.visualization.matplotlib import (
        plot_param_importances, plot_optimization_history,
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_param_importances(study, ax=axes[0])
    axes[0].set_title('Parameter Importances', fontsize=11)
    plot_optimization_history(study, ax=axes[1])
    axes[1].set_title('Optimisation History', fontsize=11)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '03_optuna_summary.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Optuna matplotlib vis unavailable: {e}')
    top = study.trials_dataframe().sort_values('value').head(10)
    print(top[['number','value','params_hidden','params_n_layers',
               'params_dropout','params_lr','params_batch_size']].to_string(index=False))


## 16 · Summary


In [ ]:
print('=' * 68)
print('LSTM SINGLE-SITE SUMMARY — Gauge 2473 (Test Set 2017–2020)')
print('=' * 68)
print(f'Best params: {best}')
print()
print(combined.to_string(index=False))
print('=' * 68)
print('LakeBeD-US seq2seq LSTM reference  DO RMSE = 1.40 mg/L')
print('Next → notebook 04_multisite_analysis.ipynb')
